# 01b FashionDB: bronze and silver

**Why:** each collection should come with a write-up. FashionDB has the Vogue Runway review of each show, from the same Vogue pages as our data, so it links to our images exactly.

**Source:** FashionDB (traopia/FashionDB on Hugging Face), by the same author as our Vogue Runway data.
- `data_vogue_final.parquet`: 14,287 shows (house, season, year, category, Vogue review, editor, image URLs, and a creative director per show)
- `final_info_designers.parquet`: 609 designers (birthplace, birth year, nationality, school)
- `final_info_fh.parquet`: 420 houses (city, founder, founding year, revenue)

Creative directors and the two smaller files are kept in bronze for the designer search later. FashionDB only covers about half our shows, so designer search needs more sources than this one.

**This notebook:**
1. Loads the three raw files into bronze, unchanged except for load details
2. Explores coverage, to see how many of our shows get a review
3. Builds a clean, validated silver table of reviews, linked to our shows by Vogue photo ID

### Download the raw files
Saves the three files from Hugging Face into the vogue Volume, next to VogueRunway.parquet. Skips any file that's already there, so re-running doesn't download again.

In [0]:
import os, urllib.request

RAW_DIR = "/Volumes/workspace/default/vogue/fashiondb"
FILES = {
    "shows": "data_vogue_final.parquet",
    "designers": "final_info_designers.parquet",
    "houses": "final_info_fh.parquet",
}

os.makedirs(RAW_DIR, exist_ok=True)
for f in FILES.values():
    path = f"{RAW_DIR}/{f}"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"https://huggingface.co/datasets/traopia/FashionDB/resolve/main/{f}", path)
print(os.listdir(RAW_DIR))

### Load raw files into bronze
Saves each file as a Delta table, unchanged except for:
- column names made safe for Delta: the houses file has names with spaces, brackets and line breaks (like "Operating revenue (Turnover) th USD 2024"), which Delta tables don't allow. They become lowercase with underscores
- `_source_file` and `_ingested_at`: where each row came from and when it was loaded

**Check:** 14,287 shows, 609 designers, 420 houses.

In [0]:
import re
from pyspark.sql import functions as F

def clean_name(c):
    return re.sub(r"[^0-9a-zA-Z]+", "_", c).strip("_").lower()

for name, f in FILES.items():
    path = f"{RAW_DIR}/{f}"
    raw = spark.read.parquet(path)
    bronze = (raw
        .select([F.col(f"`{c}`").alias(clean_name(c)) for c in raw.columns])
        .withColumn("_source_file", F.lit(path))
        .withColumn("_ingested_at", F.current_timestamp()))
    bronze.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"workspace.silhouette_archive.bronze_fashiondb_{name}")
    print(name, spark.table(f"workspace.silhouette_archive.bronze_fashiondb_{name}").count())

### Explore shows
Counts shows by year, season and category, and checks the review fields are filled, to see what FashionDB covers before linking it to our data.

In [0]:
shows = spark.table("workspace.silhouette_archive.bronze_fashiondb_shows")

display(shows.select(
    F.count("*").alias("shows"),
    F.min("year").alias("first_year"),
    F.max("year").alias("last_year"),
    F.sum((F.col("description") == "").cast("int")).alias("empty_review"),
    F.sum((F.col("editor") == "").cast("int")).alias("empty_editor"),
    F.sum((F.size("image_urls") == 0).cast("int")).alias("no_images"),
))
display(shows.groupBy("season", "category").count().orderBy("count", ascending=False))
display(shows.groupBy((F.floor(F.col("year") / 10) * 10).alias("decade")).count().orderBy("decade"))

### Explore the link to our shows
Both datasets come from Vogue Runway, so they share image URLs. Every Vogue image URL has a photo ID in it (`assets.vogue.com/photos/<photo id>/...`), which identifies the exact photo. Matching on that ID links a FashionDB show to our show without relying on house names, which are spelled differently between sources.

This counts how many of our shows (1990 to 2020) match a FashionDB show, and whether any of them match more than one.

In [0]:
PHOTO_ID = r"/photos/([0-9a-f]{24})/"

fdb_photos = (shows
    .select(F.col("index_level_0").alias("fdb_show_id"), F.explode("image_urls").alias("url"))
    .withColumn("photo_id", F.regexp_extract("url", PHOTO_ID, 1))
    .select("fdb_show_id", "photo_id").distinct())

ours = (spark.table("workspace.silhouette_archive.silver_runway_enriched")
    .filter(F.col("year").between(1990, 2020))
    .withColumn("photo_id", F.regexp_extract("url", PHOTO_ID, 1)))

show_key = ["designer", "year", "season", "line", "gender"]
links = (ours.join(fdb_photos, "photo_id")
    .groupBy(*show_key, "fdb_show_id").agg(F.count("*").alias("shared_photos")))

our_shows = ours.select(*show_key).distinct().count()
linked = links.select(*show_key).distinct().count()
multi = links.groupBy(*show_key).count().filter("count > 1").count()

print(f"Our shows: {our_shows:,}")
print(f"Linked to a FashionDB show: {linked:,} ({linked / our_shows:.0%})")
print(f"Linked to more than one: {multi:,}")
display(ours.select(*show_key).distinct()
    .join(links.select(*show_key).distinct().withColumn("linked", F.lit(1)), show_key, "left")
    .groupBy((F.floor(F.col("year") / 10) * 10).alias("decade"))
    .agg(F.count("*").alias("shows"), F.sum(F.coalesce("linked", F.lit(0))).alias("linked"))
    .orderBy("decade"))

### Explore shows that match more than one FashionDB show
541 of our shows share photos with two or more FashionDB shows. This looks at examples, with how many photos each pairing shares, to decide how silver picks one.

In [0]:
from pyspark.sql.window import Window

multi_keys = links.groupBy(*show_key).count().filter("count > 1").select(*show_key)

display(links.join(multi_keys, show_key)
    .join(shows.select(F.col("index_level_0").alias("fdb_show_id"),
                       F.col("fashion_house").alias("fdb_house"), F.col("show").alias("fdb_show"),
                       F.size("image_urls").alias("fdb_photos")), "fdb_show_id")
    .orderBy(*show_key, F.desc("shared_photos"))
    .limit(40))

# for each multi-matched show: the best pairing's shared photos vs the runner-up's
w = Window.partitionBy(*show_key).orderBy(F.desc("shared_photos"))
ranked = links.join(multi_keys, show_key).withColumn("rank", F.row_number().over(w))
display(ranked.groupBy(*show_key)
    .agg(F.max(F.when(F.col("rank") == 1, F.col("shared_photos"))).alias("best"),
         F.max(F.when(F.col("rank") == 2, F.col("shared_photos"))).alias("runner_up"))
    .select(F.sum((F.col("best") == F.col("runner_up")).cast("int")).alias("ties"),
            F.sum((F.col("runner_up") <= 2).cast("int")).alias("runner_up_2_or_fewer"),
            F.count("*").alias("multi_shows")))

### Findings: the link to our shows

- **Coverage:** 10,590 of our 21,039 shows (50%) link to a FashionDB show, so about half the archive gets a Vogue review. The rest are mostly older collections, which FashionDB covers thinly.
- **Shows matching two FashionDB shows (541):** all are 2016 in our data, and each one contains **every** photo of two Vogue shows, for example Spring 2016 and Spring 2017. Vogue's photo IDs start with an upload timestamp: the 2016 show's photos are from September 2015, the 2017 show's from September 2016. So our source labels some 2017 collections as 2016. This explains part of the 2016 spike in image counts (129,878 images in 2016, against about 80,000 in the years either side).

**Decisions:**
- **Link per image, not per show.** Each photo belongs to exactly one Vogue show, so each image takes the FashionDB show its photo ID appears in. The two shows under one 2016 label separate, and each gets its own review.
- **Take year and season from FashionDB for linked images.** They come from Vogue's own show URL. Correcting the year across the whole archive is a separate fix to the main pipeline.